# Auditoría de rumores y fichajes confirmados

## Librerías

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from math import pi
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

# Configuración visual premium
plt.style.use('dark_background')

## Carga de datos y variables

In [15]:
RUTA_DATA = Path.cwd().parent.parent / 'data'
RUTA_MERCADO = RUTA_DATA / 'scout' / 'mercado.csv'
RUTA_FILIAL = RUTA_DATA / 'laspalmasb' / 'Las Palmas Atletico 25_26.xlsx'
RUTA_UDLP = RUTA_DATA / 'udlaspalmas' / 'Individual Las Palmas 2526.xlsx'

# Perfiles Objetivo (Actualizado con siglas para el filtrado)
PERFILES_OBJETIVO = {
    'Portero': {
        'metricas': [
            'Short / medium passes per 90', 'Forward passes per 90', 
            'Long passes per 90', 'Conceded goals per 90', 'Prevented goals per 90'
        ],
        'pesos': [0.15, 0.15, 0.15, 0.25, 0.30],
        'siglas': 'GK'
    },
    'Defensa central': {
        'metricas': [
            'Duelos_Defensivos_P90', 'Interceptaciones_P90', 'Duelos_Aereos_P90',
            'Pases_Ultimo_Tercio_P90', 'Conducciones_Progresivas_P90'
        ],
        'pesos': [0.30, 0.25, 0.20, 0.15, 0.10],
        'siglas': 'CB'
    },
    'Lateral': {
        'metricas': [
            'Conducciones_Progresivas_P90', 'Centros_P90', 'Pases_Ultimo_Tercio_P90',
            'Duelos_Defensivos_P90', 'Interceptaciones_P90'
        ],
        'pesos': [0.25, 0.20, 0.25, 0.15, 0.15],
        'siglas': 'RB|LB'
    },
    'Mediocentro': {
        'metricas': [
            'Pases_Ultimo_Tercio_P90', 'Forward passes per 90',
            'Duelos_Defensivos_P90', 'Interceptaciones_P90', 'Conducciones_Progresivas_P90'
        ],
        'pesos': [0.30, 0.20, 0.25, 0.15, 0.10],
        'siglas': 'CMF|DMF'
    },
    'Mediocentro ofensivo': {
        'metricas': [
            'Pases_Ultimo_Tercio_P90', 'Pases_Clave_P90', 
            'Conducciones_Progresivas_P90', 'Regates_P90', 'xA_P90'
        ],
        'pesos': [0.30, 0.25, 0.20, 0.15, 0.10],
        'siglas': 'AMF'
    },
    'Extremo': {
        'metricas': [
            'Regates_P90', 'Conducciones_Progresivas_P90', 'Centros_P90', 
            'xA_P90', 'Toques_Area_P90'
        ],
        'pesos': [0.30, 0.25, 0.15, 0.15, 0.15],
        'siglas': 'RW|LW'
    },
    'Delantero centro': {
        'metricas': [
            'Goles_P90', 'xG_P90', 'Toques_Area_P90', 
            'Remates_P90', 'Duelos_Ofensivos_P90'
        ],
        'pesos': [0.35, 0.25, 0.20, 0.10, 0.10],
        'siglas': 'CF'
    }
}

DICCIONARIO_COLUMNAS = {
    'Player': 'Jugador', 'Team': 'Equipo', 'Position': 'Posicion', 'Age': 'Edad', 
    'Market value': 'Valor_Mercado', 'Contract expires': 'Fin_Contrato', 
    'Minutes played': 'Minutos', 'Team within selected timeframe': 'Equipo_Actual',
    'Defensive duels per 90': 'Duelos_Defensivos_P90', 'Duelos defensivos/90': 'Duelos_Defensivos_P90',
    'Interceptions per 90': 'Interceptaciones_P90', 'Interceptaciones/90': 'Interceptaciones_P90',
    'Aerial duels per 90': 'Duelos_Aereos_P90', 'Duelos aéreos en los 90': 'Duelos_Aereos_P90',
    'Passes to final third per 90': 'Pases_Ultimo_Tercio_P90', 'Pases en el último tercio/90': 'Pases_Ultimo_Tercio_P90',
    'Progressive runs per 90': 'Conducciones_Progresivas_P90', 'Carreras en progresión/90': 'Conducciones_Progresivas_P90',
    'Key passes per 90': 'Pases_Clave_P90', 'Jugadas claves/90': 'Pases_Clave_P90',
    'Dribbles per 90': 'Regates_P90', 'Regates/90': 'Regates_P90',
    'Crosses per 90': 'Centros_P90', 'Centros/90': 'Centros_P90',
    'xA per 90': 'xA_P90', 'xA/90': 'xA_P90',
    'Goals per 90': 'Goles_P90', 'Goles/90': 'Goles_P90',
    'xG per 90': 'xG_P90', 'xG/90': 'xG_P90',
    'Touches in box per 90': 'Toques_Area_P90', 'Toques en el área de penalti/90': 'Toques_Area_P90',
    'Shots per 90': 'Remates_P90', 'Remates/90': 'Remates_P90',
    'Offensive duels per 90': 'Duelos_Ofensivos_P90', 'Duelos atacantes/90': 'Duelos_Ofensivos_P90'
}

In [16]:
# Cargar el Mercado General
try:
    df_mercado = pd.read_csv(RUTA_MERCADO).fillna(0)
    df_mercado['Base_Origen'] = 'Mercado Exterior'
except Exception as e:
    print(f"Error cargando mercado.csv: {e}")
    df_mercado = pd.DataFrame()

# Cargar el Filial y normalizarlo
try:
    df_filial = pd.read_excel(RUTA_FILIAL).fillna(0)
    df_filial = df_filial.rename(columns=DICCIONARIO_COLUMNAS)
    df_filial['Base_Origen'] = 'Las Palmas Atlético'
    df_filial['Carpeta_Origen'] = 'Filial'
except Exception as e:
    print(f"Error cargando filial: {e}")
    df_filial = pd.DataFrame()

# Cargar la Plantilla Actual UDLP
try:
    df_udlp = pd.read_excel(RUTA_UDLP).fillna(0)
    df_udlp = df_udlp.rename(columns=DICCIONARIO_COLUMNAS)
except Exception as e:
    print(f"Error cargando plantilla UDLP: {e}")
    df_udlp = pd.DataFrame()

# UNIFICACIÓN: Mercado + Filial en un solo Data Lake
columnas_comunes = list(set(df_mercado.columns) & set(df_filial.columns))
df_global = pd.concat([df_mercado[columnas_comunes], df_filial[columnas_comunes]], ignore_index=True)
df_global = df_global.drop_duplicates(subset=['Jugador', 'Edad'], keep='first').reset_index(drop=True)

print(f"Universo Global (Mercado + Filial) consolidado: {len(df_global)} jugadores.")
print(f"Plantilla UDLP cargada: {len(df_udlp)} jugadores.")

Universo Global (Mercado + Filial) consolidado: 7323 jugadores.
Plantilla UDLP cargada: 27 jugadores.


## Funciones de similitud

In [17]:
def calcular_ajuste_estilo(jugador_row, clave_perfil, df_contexto):
    """Calcula el % de encaje del jugador contra el estilo UDLP normalizado a su grupo"""
    perfil = PERFILES_OBJETIVO.get(clave_perfil)
    if not perfil: return 0
    
    metricas = perfil['metricas']
    pesos = perfil['pesos']
    siglas = perfil['siglas']
    
    # Filtramos a todos los de su posición para la normalización
    df_pos = df_contexto[df_contexto['Posicion'].str.contains(siglas, case=False, na=False)].copy()
    if df_pos.empty: return 0
    
    score_final = 0
    for metrica, peso in zip(metricas, pesos):
        if metrica in df_pos.columns and metrica in jugador_row.index:
            minimo = df_pos[metrica].min()
            maximo = df_pos[metrica].max()
            rango = maximo - minimo
            
            valor_jugador = jugador_row[metrica]
            valor_norm = 0.5 if rango == 0 else (valor_jugador - minimo) / rango
            score_final += valor_norm * peso
            
    return round(score_final * 100, 2)

def visualizar_comparativa_radar(jugador_ext, jugador_udlp, clave_perfil):
    """Genera un Radar Chart comparando al fichaje vs el jugador de la plantilla"""
    perfil = PERFILES_OBJETIVO[clave_perfil]
    metricas = perfil['metricas']
    
    # Extraemos métricas
    valores_ext = jugador_ext[metricas].values.flatten().tolist()
    valores_udlp = jugador_udlp[metricas].values.flatten().tolist()
    
    # Normalización para el gráfico (escalado sobre el máximo de ambos)
    maximos = [max(e, u) if max(e, u) > 0 else 1 for e, u in zip(valores_ext, valores_udlp)]
    valores_ext_norm = [e / m for e, m in zip(valores_ext, maximos)] + [valores_ext[0] / maximos[0]]
    valores_udlp_norm = [u / m for u, m in zip(valores_udlp, maximos)] + [valores_udlp[0] / maximos[0]]
    
    # Geometría del Radar
    N = len(metricas)
    angulos = [n / float(N) * 2 * pi for n in range(N)] + [0]
    nombres_cortos = [m.replace('_P90', '').replace('_', ' ') for m in metricas]
    
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    plt.xticks(angulos[:-1], nombres_cortos, color='white', size=11, fontweight='bold')
    ax.set_rlabel_position(0)
    plt.yticks([], []) # Ocultar grid numérico
    
    # Trazar Fichaje (Amarillo UDLP)
    ax.plot(angulos, valores_ext_norm, linewidth=2.5, linestyle='solid', color='#FFD700', label=f"{jugador_ext['Jugador']} (Rumor/Fichaje)")
    ax.fill(angulos, valores_ext_norm, color='#FFD700', alpha=0.3)
    
    # Trazar Jugador UDLP (Azul UDLP)
    ax.plot(angulos, valores_udlp_norm, linewidth=2.5, linestyle='--', color='#005BBB', label=f"{jugador_udlp['Jugador']} (Plantilla)")
    ax.fill(angulos, valores_udlp_norm, color='#005BBB', alpha=0.2)
    
    plt.title(f"Comparativa Táctica: {jugador_ext['Jugador']} vs {jugador_udlp['Jugador']}", size=16, color='white', pad=30, fontweight='bold')
    plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.show()

## Visualizaciones

In [18]:
# Crear elementos del menú
output_visual = widgets.Output()

# Filtros
ligas = ['Todas'] + sorted(df_global['Equipo_Actual'].dropna().unique().tolist())
dropdown_liga = widgets.Dropdown(options=ligas, description='Liga:', style={'description_width': 'initial'})
dropdown_club = widgets.Dropdown(options=['Todos'], description='Club:', style={'description_width': 'initial'})
dropdown_jugador_mercado = widgets.Dropdown(options=[], description='Fichaje/Rumor:', style={'description_width': 'initial'})
dropdown_perfil = widgets.Dropdown(options=list(PERFILES_OBJETIVO.keys()), description='Rol Táctico:', style={'description_width': 'initial'})
dropdown_jugador_udlp = widgets.Dropdown(options=sorted(df_udlp['Jugador'].dropna().unique().tolist()), description='Comparar con (UDLP):', style={'description_width': 'initial'})

boton_analizar = widgets.Button(description='Auditar Fichaje', button_style='warning', icon='search')

# Lógica de actualización en cascada
def actualizar_clubes(*args):
    liga_sel = dropdown_liga.value
    if liga_sel == 'Todas':
        clubes = ['Todos'] + sorted(df_global['Equipo'].dropna().unique().tolist())
    else:
        clubes = ['Todos'] + sorted(df_global[df_global['Equipo_Actual'] == liga_sel]['Equipo'].dropna().unique().tolist())
    dropdown_club.options = clubes
    
def actualizar_jugadores(*args):
    liga_sel = dropdown_liga.value
    club_sel = dropdown_club.value
    
    df_temp = df_global.copy()
    if liga_sel != 'Todas': df_temp = df_temp[df_temp['Equipo_Actual'] == liga_sel]
    if club_sel != 'Todos': df_temp = df_temp[df_temp['Equipo'] == club_sel]
        
    jugadores = sorted(df_temp['Jugador'].dropna().unique().tolist())
    dropdown_jugador_mercado.options = jugadores

dropdown_liga.observe(actualizar_clubes, 'value')
dropdown_club.observe(actualizar_jugadores, 'value')

# Lógica del botón de Análisis
def on_boton_click(b):
    with output_visual:
        clear_output(wait=True)
        nombre_fichaje = dropdown_jugador_mercado.value
        nombre_udlp = dropdown_jugador_udlp.value
        clave_perfil = dropdown_perfil.value
        
        if not nombre_fichaje or not nombre_udlp:
            print("Selecciona un jugador del mercado y uno de la UDLP.")
            return
            
        jugador_ext = df_global[df_global['Jugador'] == nombre_fichaje].iloc[0]
        jugador_loc = df_udlp[df_udlp['Jugador'] == nombre_udlp].iloc[0]
        
        print(f"{'='*80}")
        print(f"INFORME EJECUTIVO DE SCOUTING: {nombre_fichaje.upper()}")
        print(f"{'='*80}")
        print(f"Origen: {jugador_ext['Equipo']} | Edad: {jugador_ext['Edad']} años | Valor: {jugador_ext.get('Valor_MEur', 'N/A')}M€")
        
        # Ajuste al Estilo
        score = calcular_ajuste_estilo(jugador_ext, clave_perfil, df_global)
        print(f"SCORE DE ENCAJE EN EL ADN UDLP ({clave_perfil}): {score}/100")
        
        if score > 80: print("🟢 VALORACIÓN: Fichaje Altamente Recomendado para el Sistema.")
        elif score > 60: print("🟡 VALORACIÓN: Fichaje Viable. Requiere adaptación táctica.")
        else: print("🔴 VALORACIÓN: Riesgo Táctico. No encaja con los parámetros del modelo.")
        
        # Radar de Comparación
        visualizar_comparativa_radar(jugador_ext, jugador_loc, clave_perfil)
        
        # Dashboard: El Panorama del Mercado
        metricas = PERFILES_OBJETIVO[clave_perfil]['metricas']
        siglas = PERFILES_OBJETIVO[clave_perfil]['siglas']
        
        df_panorama = df_global[df_global['Posicion'].str.contains(siglas, case=False, na=False)].copy()
        
        # Gráfico Scatter (Edad vs Principal Métrica de la Posición)
        metrica_clave = metricas[0] 
        
        plt.figure(figsize=(12, 6))
        sns.scatterplot(data=df_panorama, x='Edad', y=metrica_clave, color='gray', alpha=0.3, label='Resto del Mercado')
        
        # Destacar al fichaje evaluado
        plt.scatter(jugador_ext['Edad'], jugador_ext[metrica_clave], color='#FFD700', s=200, edgecolor='white', zorder=5, label=nombre_fichaje)
        
        # Destacar al jugador actual de la UDLP (buscando sus datos en la plantilla)
        if metrica_clave in jugador_loc:
            plt.scatter(jugador_loc['Edad'], jugador_loc[metrica_clave], color='#005BBB', s=200, edgecolor='white', zorder=5, label=nombre_udlp)
            
        plt.title(f'Panorama de Mercado: {clave_perfil} ({metrica_clave.replace("_P90", "")} vs Edad)', fontsize=14, pad=15)
        plt.legend()
        plt.show()

boton_analizar.on_click(on_boton_click)

# Iniciar opciones iniciales
actualizar_clubes()
actualizar_jugadores()

# Mostrar Interfaz
display(widgets.VBox([
    widgets.HBox([dropdown_liga, dropdown_club]),
    widgets.HBox([dropdown_jugador_mercado, dropdown_perfil]),
    widgets.HBox([dropdown_jugador_udlp]),
    boton_analizar,
    output_visual
]))